## Setup

### Environment

In [ ]:
!pip install torch
!pip install wandb
!pip install vllm -q
!pip install convokit
!pip install torchvision
!pip install transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 101.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
convokit 3.1.0 requires numpy>=2.0.0, but you have numpy 1.26.4 which is incompatible.
en-core-web-sm 3.7.1 requires spacy<3.8.0,>=3.7.2, but you have spacy 3.8.4 which is incompatible.
gensim 4.3.3 requires scipy<1.14.0,>=1.7.0, but you have scipy 1.15.2 which is incompatible.
  Using cached numpy-2.2.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
  Using cached numpy-2.0.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (60 kB)
Using cached numpy-2.0.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (19.5 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninst

In [ ]:
#imports
import sentencepiece as spm
import pandas as pd
import numpy as np

import string
import wandb
import torch
import time
import math
import re

from tqdm import tqdm
from vllm import LLM
from torch.utils.data import Dataset
from convokit import Corpus, download
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, DataCollatorWithPadding, TrainingArguments, pipeline

INFO 02-18 14:55:36 __init__.py:190] Automatically detected platform cuda.


In [ ]:
# Determine device capabilities
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Login to wandb
wandb.login()

<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: hawfi004 (hawfi004-university-of-minnesota) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

### Handling Data

#### Import and Clean

In [ ]:
# Import data and convert to dataframe
wiki_corpus = Corpus(filename=download("wikipedia-politeness-corpus"))
se_corpus = Corpus(filename=download("stack-exchange-politeness-corpus"))

wiki_df = wiki_corpus.get_utterances_dataframe()[['text', 'meta.Binary']].rename(columns={'meta.Binary': 'target'})
test_df = se_corpus.get_utterances_dataframe()[['text', 'meta.Binary']].rename(columns={'meta.Binary': 'target'})

train_df = wiki_df.sample(frac=0.8, random_state=25)
train_df['target'] = train_df['target'] + 1

eval_df = wiki_df.drop(train_df.index)
eval_df['target'] = eval_df['target'] + 1

test_df['target'] = test_df['target'] + 1

No configuration file found at /root/.convokit/config.yml; writing with contents: 
# Default Backend Parameters
db_host: localhost:27017
data_directory: ~/.convokit/saved-corpora
model_directory: ~/.convokit/saved-models
default_backend: mem


In [ ]:
# Clean Text

def clean_text(text):
  '''Make text lowercase, remove text in square brackets,remove links,remove punctuation
    and remove words containing numbers.'''
  text = str(text).lower()
  text = re.sub('\[.*?\]', '', text)
  text = re.sub('https?://\S+|www\.\S+', '', text)
  text = re.sub('<.*?>+', '', text)
  text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
  text = re.sub('\n', '', text)
  text = re.sub('\w*\d\w*', '', text)
  return text

train_df['text_clean'] = train_df['text'].apply(clean_text)
eval_df['text_clean'] = eval_df['text'].apply(clean_text)
test_df['text_clean'] = test_df['text'].apply(clean_text)

#### Encoding

In [ ]:
# Define byte-pair encoding to create a predefined vocabulary

# Create directory for preprocessed text
!mkdir preprocessed_data
with open('preprocessed_data/train.txt', 'w') as f:
  for text in train_df['text_clean']:
    f.write(f'{text.strip()}\n')

VOCAB_SIZE = 8000
# Learn a BPE tokenization model
spm.SentencePieceTrainer.train(input='preprocessed_data/train.txt', model_prefix='bpe_model', vocab_size=VOCAB_SIZE, model_type='bpe')

sp = spm.SentencePieceProcessor(model_file='bpe_model.model')

def tokenize_into_str(text):
  return ' '.join(sp.encode(text, out_type=str))

def tokenize_into_idx(text):
  return ' '.join([str(tok) for tok in sp.encode(text)])

In [ ]:
# Encode training data
train_df['text_tokenized'] = train_df['text_clean'].apply(tokenize_into_str)
train_df['text_indices'] = train_df['text_clean'].apply(tokenize_into_idx)
train_df['text_indices'] = train_df['text_indices'].replace('', np.nan)
train_df.dropna(subset=['text_indices'], inplace=True)

# Encode eval data
eval_df['text_tokenized'] = eval_df['text_clean'].apply(tokenize_into_str)
eval_df['text_indices'] = eval_df['text_clean'].apply(tokenize_into_idx)
eval_df['text_indices'] = eval_df['text_indices'].replace('', np.nan)
eval_df.dropna(subset=['text_indices'], inplace=True)

# Encode testing data
test_df['text_tokenized'] = test_df['text_clean'].apply(tokenize_into_str)
test_df['text_indices'] = test_df['text_clean'].apply(tokenize_into_idx)
test_df['text_indices'] = test_df['text_indices'].replace('', np.nan)
test_df.dropna(subset=['text_indices'], inplace=True)

In [ ]:
# Check Sequence Length
def count_tokens(text_indices):
  tokens = text_indices.split(' ')
  return len(tokens)

train_df['sequence_length'] = train_df['text_indices'].apply(count_tokens)
eval_df['sequence_length'] = eval_df['text_indices'].apply(count_tokens)
test_df['sequence_length'] = test_df['text_indices'].apply(count_tokens)

#### Saving and Final Preparation

In [ ]:
# save files
train_df.to_pickle('preprocessed_data/train.pkl')
eval_df.to_pickle('preprocessed_data/eval.pkl')
test_df.to_pickle('preprocessed_data/test.pkl')

In [ ]:
# Creating a custom dataset for consumption
class PoliteDataset(Dataset):
  def __init__(self, pickle_path):
    self.dataset = pd.read_pickle(pickle_path)

  def __len__(self):
    return len(self.dataset)

  def __getitem__(self, idx):
    text_indices_string = self.dataset['text_indices'].iloc[idx]
    text_indices = [int(tok_idx) for tok_idx in text_indices_string.split(' ')]

    attention_mask = [1] * len(text_indices)
    sequence_length = int(self.dataset['sequence_length'].iloc[idx])
    label = None
    if 'target' in self.dataset.columns:
      label = int(self.dataset['target'].iloc[idx])
    return {'input_ids': text_indices, 'attention_mask': attention_mask, 'labels': label}

train_dataset = PoliteDataset('preprocessed_data/train.pkl')
eval_dataset = PoliteDataset('preprocessed_data/eval.pkl')
test_dataset = PoliteDataset('preprocessed_data/test.pkl')

In [ ]:
# Batching
def generate_batch(batch):
  batch_indices = []
  batch_labels = []
  offsets = [0]

  for text_indices, sequence_length, label in batch:
    batch_indices.extend(text_indices)
    batch_labels.append(label)
    offsets.append(sequence_length)

  batch_indices = torch.tensor(batch_indices, dtype=torch.long)
  batch_labels = torch.tensor(batch_labels, dtype=torch.long)
  offsets = torch.tensor(offsets[:-1]).cumsum(dim=0)
  return batch_indices, offsets, batch_labels

In [ ]:
# Final variables and reproduceability
BATCH_SIZE = 8
NUM_EPOCHS = 10

SEED = 42
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

### Model Creation

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("cardiffnlp/twitter-roberta-base-sentiment-latest")
model = AutoModelForSequenceClassification.from_pretrained("cardiffnlp/twitter-roberta-base-sentiment-latest")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


## Execution

In [ ]:
# Define custom trainer
class CustomTrainer(Trainer):
  def _inner_training_loop(
      self, batch_size=None, args=None, resume_from_checkpoint=None, trial=None, ignore_keys_for_eval=None
  ):
    number_of_epochs = args.num_train_epochs
    start = time.time()
    train_loss = []
    train_acc = []
    eval_acc = []

    # criterion = torch.nn.CrossEntropyLoss()
    criterion = torch.nn.CrossEntropyLoss().to(device)
    self.optimizer = torch.optim.Adam(self.model.parameters(), lr=args.learning_rate)
    self.scheduler = torch.optim.lr_scheduler.StepLR(self.optimizer, step_size=1, gamma=0.9)

    train_dataloader = self.get_train_dataloader()
    eval_dataloader = self.get_eval_dataloader()

    max_steps = math.ceil(args.num_train_epochs * len(train_dataloader))

    for epoch in range(number_of_epochs):
      # Training
      train_loss_per_epoch = 0
      train_acc_per_epoch = 0
      self.model.train()
      with tqdm(train_dataloader, unit="batch") as training_epoch:
        training_epoch.set_description(f"Training Epoch {epoch}")
        for step, inputs in enumerate(training_epoch):
          inputs = inputs.to(device)
          labels = inputs['labels']

          # forward pass
          self.optimizer.zero_grad()

          # TODO implement output, get and calculate loss
          output = self.model(**inputs)
          logits = output['logits']
          loss = criterion(logits, labels)

          train_loss_per_epoch += loss.item()

          # calculate gradients
          loss.backward()
          # update weights
          self.optimizer.step()
          train_acc_per_epoch += (output['logits'].argmax(1) == labels).sum().item()

      # adjust learning rate
      self.scheduler.step()
      train_loss_per_epoch /= len(train_dataloader)
      train_acc_per_epoch /= (len(train_dataloader) * batch_size)

      # Eval
      eval_loss_per_epoch = 0
      eval_acc_per_epoch = 0
      # self.model.eval()
      with tqdm(eval_dataloader, unit="batch") as eval_epoch:
        eval_epoch.set_description(f"Evaluation Epoch {epoch}")
        #TODO implement
        for step, inputs in enumerate(eval_epoch):
          inputs = inputs.to(device)
          labels = inputs['labels']

          # forward pass
          self.optimizer.zero_grad()
          # TODO implement output, get and calculate loss
          output = self.model(**inputs)
          logits = output['logits']
          loss = criterion(logits, labels)

          eval_loss_per_epoch += loss.item()

          # calculate gradients
          # loss.backward()
          # update weights
          # self.optimizer.step()
          eval_acc_per_epoch += (output['logits'].argmax(1) == labels).sum().item()

      # adjust learning rate
      # self.scheduler.step()
      eval_loss_per_epoch /= len(eval_dataloader)
      eval_acc_per_epoch /= (len(eval_dataloader) * batch_size)

      # Print Findings
      print(f'\tTrain Loss: {train_loss_per_epoch:.3f} | Train Acc: {train_acc_per_epoch*100:.2f}')
      print(f'\tEval Loss: {eval_loss_per_epoch:.3f} | Eval Acc: {eval_acc_per_epoch*100:.2f}')

      wandb.log({"train_accuracy": train_acc_per_epoch, "train_loss": train_loss_per_epoch}, step = epoch)
      wandb.log({"eval_accuracy": eval_acc_per_epoch, "eval_loss": eval_loss_per_epoch}, step = epoch)

    print(f'Time: {(time.time() - start)/60:.3f} minutes')

In [ ]:
# Setup wandb project
run = wandb.init(
    project = "Politeness RoBERTa Classifier",
    config = {
        "epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE
        }
)

# Establish desired metrics
def compute_metrics(eval_pred):
  logits, labels = eval_pred
  predictions = np.argmax(logits, axis=-1)
  return {"accuracy": (predictions == labels).mean()}

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

In [ ]:
# Training
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
training_args = TrainingArguments(
    output_dir = 'results',
    num_train_epochs = NUM_EPOCHS
    )
trainer = CustomTrainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    processing_class = tokenizer,
    data_collator = data_collator,
    compute_metrics = compute_metrics
)
trainer.train()

Evaluation Epoch 0: 100%|██████████| 109/109 [00:02<00:00, 38.69batch/s]


	Train Loss: 1.056 | Train Acc: 49.57
	Eval Loss: 1.056 | Eval Acc: 48.97


Evaluation Epoch 1: 100%|██████████| 109/109 [00:03<00:00, 35.97batch/s]


	Train Loss: 1.046 | Train Acc: 50.17
	Eval Loss: 1.068 | Eval Acc: 48.97


Evaluation Epoch 2: 100%|██████████| 109/109 [00:03<00:00, 35.38batch/s]


	Train Loss: 1.047 | Train Acc: 50.14
	Eval Loss: 1.053 | Eval Acc: 48.97


Training Epoch 3:  74%|███████▎  | 320/435 [00:33<00:13,  8.85batch/s]

In [ ]:
# Testing
# trainer.evaluate(test_dataset)

with tqdm(eval_dataloader, unit="batch") as eval_epoch:
  eval_epoch.set_description(f"Evaluation Epoch {epoch}")
  #TODO implement
  for step, inputs in enumerate(eval_epoch):
    inputs = inputs.to(device)
    labels = inputs['labels']

    # forward pass
    self.optimizer.zero_grad()
    # TODO implement output, get and calculate loss
    output = self.model(**inputs)
    logits = output['logits']
    loss = criterion(logits, labels)

    eval_loss_per_epoch += loss.item()
    eval_acc_per_epoch += (output['logits'].argmax(1) == labels).sum().item()

# adjust learning rate
# self.scheduler.step()
eval_loss_per_epoch /= len(eval_dataloader)
eval_acc_per_epoch /= (len(eval_dataloader) * batch_size)

# Print Findings
print(f'\tTrain Loss: {train_loss_per_epoch:.3f} | Train Acc: {train_acc_per_epoch*100:.2f}')
print(f'\tEval Loss: {eval_loss_per_epoch:.3f} | Eval Acc: {eval_acc_per_epoch*100:.2f}')

In [ ]:
# Create LLM Model
llm = LLM(model="jason9693/Qwen2.5-1.5B-apeach", dtype="half", task="classify")

In [ ]:
prompt_prelude = '''Task: Given text input, predict whether or not it is inherently
                    polite or impolite. The predicted label should be -1 if the sentiment
                    is impolite, and 1 if the sentiment is polite.\n
            Input: '''
texts = test_dataset.dataset['text']
targets = test_dataset.dataset['target']
prompts = [prompt_prelude + text for text in texts]
outputs = llm.classify(prompts)

In [ ]:
def output_to_label(output):
  probs = output.outputs.probs
  label = int(np.argmax(probs))
  label = - 1 if label != 1 else label
  return label

outputs_labels = list(map(output_to_label, outputs))

incorrect_ids = []
correct_ids = []
for i in range(len(outputs_labels)):
  if outputs_labels[i] != targets[i]:
    incorrect_ids.append(i)
  else:
    correct_ids.append(i)

acc = len(correct_ids) / len(targets)
print(f'\tAccuracy: {acc*100:.2f}')